# Week 14 — Offline RL Workbook

IQL, CQL, behaviour regularisation, and a tiny SNIPS off-policy evaluator.

**Teaching companion:** [Worked examples, derivations, analogies, and self-checks](teaching_guide.md). Read the relevant example before attempting its exercise.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('.'))
from common.plotting import use_rl_style, RL_PALETTE
from common.seed import seed_everything
use_rl_style(); seed_everything(0)
import numpy as np, matplotlib.pyplot as plt
from project import solutions as S

## Exercise 1 (★) — Expectile loss

> **Hint** (try yourself first): The expectile loss is `|tau - 1{u<0}| * u^2`. For `tau > 0.5` it penalises positive residuals (under-prediction) more heavily.

In [ ]:
def expectile_loss(u, tau):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
expectile_loss = S.exercise_1_expectile
u = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])
out = expectile_loss(u, 0.8)
assert out.shape == u.shape and (out >= 0).all() and out[-1] > out[0]

## Exercise 2 (★) — AWR weights

> **Hint**: $w_i \propto \exp(\beta A_i)$; cap raw weights before normalising; shift log weights to avoid overflow; normalise so the weights average to 1.

In [ ]:
def awr_weights(adv, beta):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
awr_weights = S.exercise_2_awr_weights
w = awr_weights(np.array([-1.0, 0.0, 1.0]), beta=2.0)
assert abs(w.mean() - 1.0) < 1e-6

## Exercise 3 (★★) — CQL penalty

> **Hint**: For a fixed state $s$, the penalty is `logsumexp(Q[s,:]) - Q[s, a_beta]`. For two or more finite Q-values it is strictly positive. It tends to zero only as the logged action dominates the softmax (or equals zero with a single available action).

In [ ]:
def cql_penalty(Q_s, a_beta):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
cql_penalty = S.exercise_3_cql_penalty
Q = np.array([1.0, 0.0, 0.0, 0.0])
assert cql_penalty(Q, 0) < cql_penalty(Q, 3)

## Exercise 4 (★★) — Collect the logged dataset

> **Hint**: Use the `collect_logged_dataset` helper. Inspect coverage by counting visits per (s,a). `episode_end` marks collection boundaries; `done` marks genuine termination for Bellman targets.

In [ ]:
def make_data(n_eps=80, seed=0):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
make_data = S.exercise_4_collect_dataset
data = make_data(50)
assert data['s_idx'].shape == data['a'].shape

## Exercise 5 (★★) — Train IQL

> **Hint**: Use the reference `train` helper with `algo='iql'`. Plot the evaluation return curve.

In [ ]:
def train_iql(seed=0, n_steps=1200):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
train_iql = S.exercise_5_train_iql
out = train_iql(0, 500)
assert 'final_return' in out

## Exercise 6 (★★) — CQL ablation over $\alpha$

> **Hint**: Sweep $\alpha \in \{0.1, 1, 10\}$ and plot final return vs $\alpha$.

In [ ]:
def cql_ablation(alphas, n_steps=500):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def cql_ablation(alphas, n_steps=500):
    return {a: S.exercise_6_train_cql(0, n_steps, a)['final_return'] for a in alphas}
res = cql_ablation([0.1, 1.0, 10.0], 300)
assert set(res.keys()) == {0.1, 1.0, 10.0}

## Exercise 7 (★★★) — SNIPS off-policy evaluation

> **Hint**: For each episode, accumulate per-step likelihood ratio `pi(a|s)/pi_beta(a|s)` and the discounted return. Split on `episode_end` and accumulate weights in log space. Then `sum(w*G)/sum(w)`; this evaluates discounted episode return for the explicitly smoothed target policy.

In [ ]:
def snips_estimate(data, target_actions, gamma=0.95):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
snips_estimate = S.exercise_7_off_policy_evaluation_is
data = S.exercise_4_collect_dataset(50)
target = np.argmax(data['Q_star'], axis=1)
est = snips_estimate(data, target)
assert np.isfinite(est)

## Exercise 8 (★★★, meaningful objective) — Compare three algorithms

This exercise mirrors the recruitment-matching question: if you only had logged recruiter decisions and outcomes, which offline algorithm would you trust to rank candidates? Run all three on the same dataset and report final returns.

In [ ]:
def compare(seed=0, n_steps=800):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
compare = S.exercise_8_compare_algorithms
res = compare(0, 400)
print(res)
assert set(res.keys()) == {'naive', 'cql', 'iql'}

<!-- AUTO-HINTS-SECTION -->
## Hints toward answers

Attempt the exercise first. Before opening a hint, write the quantity you are
estimating, the data source, and the expected update direction. The
[teaching guide](teaching_guide.md) supplies worked examples, assumptions,
and transfer questions. The hints below collect the exercise's own inline
hints; they do not reveal or execute reference solutions.

<details>
<summary>Exercise 1 (★) — Expectile loss</summary>

> **Hint** (try yourself first): The expectile loss is `|tau - 1{u<0}| * u^2`. For `tau > 0.5` it penalises positive residuals (under-prediction) more heavily.

</details>

<details>
<summary>Exercise 2 (★) — AWR weights</summary>

> **Hint**: $w_i \propto \exp(\beta A_i)$; cap raw weights before normalising; shift log weights to avoid overflow; normalise so the weights average to 1.

</details>

<details>
<summary>Exercise 3 (★★) — CQL penalty</summary>

> **Hint**: For a fixed state $s$, the penalty is `logsumexp(Q[s,:]) - Q[s, a_beta]`. For two or more finite Q-values it is strictly positive. It tends to zero only as the logged action dominates the softmax (or equals zero with a single available action).

</details>

<details>
<summary>Exercise 4 (★★) — Collect the logged dataset</summary>

> **Hint**: Use the `collect_logged_dataset` helper. Inspect coverage by counting visits per (s,a). `episode_end` marks collection boundaries; `done` marks genuine termination for Bellman targets.

</details>

<details>
<summary>Exercise 5 (★★) — Train IQL</summary>

> **Hint**: Use the reference `train` helper with `algo='iql'`. Plot the evaluation return curve.

</details>

<details>
<summary>Exercise 6 (★★) — CQL ablation over $\alpha$</summary>

> **Hint**: Sweep $\alpha \in \{0.1, 1, 10\}$ and plot final return vs $\alpha$.

</details>

<details>
<summary>Exercise 7 (★★★) — SNIPS off-policy evaluation</summary>

> **Hint**: For each episode, accumulate per-step likelihood ratio `pi(a|s)/pi_beta(a|s)` and the discounted return. Split on `episode_end` and accumulate weights in log space. Then `sum(w*G)/sum(w)`; this evaluates discounted episode return for the explicitly smoothed target policy.

</details>

<details>
<summary>Exercise 8 (★★★, meaningful objective) — Compare three algorithms</summary>

No inline hint is supplied for this exercise. Use the corresponding derivation and code bridge in the teaching guide.

</details>

After solving, change one input or assumption and predict the result before
running it. If a cell calls `project.solutions`, passing it verifies the
reference demonstration; test your own implementation separately.
